# 01 — Simple model

First-pass model of a hydrodynamically focused flow cytometer channel and the ADC sample rate it requires.

Workflow: derive/try things here, and once a formula stabilises move it into `src/cytosim/`.
`autoreload` picks up edits to the package without restarting the kernel.

**Units:** `Params` fields are stored in SI. Type lengths as `um(...)` and flow rates as `ul_per_s(...)` / `ml_per_min(...)`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from cytosim import Params, simulate, units
from cytosim.units import um, ul_per_s, ml_per_min
from cytosim.params import MILK_PARTICLES, MEASURED_CORE_WIDTH_RANGE
from cytosim import geometry, flow, optics, particle, signal


def report(r):
    print(f"core diameter   : {units.to_um(r.core_diameter):8.1f} µm   (measured {units.to_um(MEASURED_CORE_WIDTH_RANGE[0]):.0f}-{units.to_um(MEASURED_CORE_WIDTH_RANGE[1]):.0f} µm)")
    print(f"core velocity   : {r.velocity:8.2f} m/s")
    print(f"Reynolds        : {r.reynolds:8.0f}")
    print(f"pulse FWHM      : {r.fwhm*1e9:8.0f} ns")
    print(f"pulse sigma_t   : {r.sigma_t*1e9:8.0f} ns")
    print(f"f_3dB           : {r.f_3db/1e6:8.2f} MHz")
    print(f"f_s (FWHM crit.): {r.f_s_fwhm/1e6:8.2f} MHz")
    print(f"f_s (BW crit.)  : {r.f_s_bandwidth/1e6:8.2f} MHz")
    print(f"f_s required    : {r.f_s_required/1e6:8.2f} MHz")

## Instrument defaults

200 × 200 µm channel, sheath 20 mL/min, sample 10 µL/s (EB, 1:1) or 7 µL/s (PR2, 1:3.2), 20 × 100 µm spot.

In [ ]:
p = Params.eb()
p

In [ ]:
for name, preset in (("EB 1:1", Params.eb), ("PR2 1:3.2", Params.pr2)):
    print(f"=== {name} ===")
    report(simulate(preset()))
    print()

## Plug vs parabolic focusing

Plug flow overestimates the core width (38 µm vs the measured 10–30 µm); a parabolic profile (core at 2× mean velocity) gives 27 µm. That is why `velocity_profile="parabolic"` is the default — and it doubles the required sample rate.

In [ ]:
for prof in ("plug", "parabolic"):
    r = simulate(Params(velocity_profile=prof))
    print(f"{prof:10s} core {units.to_um(r.core_diameter):5.1f} µm   v {r.velocity:5.2f} m/s   f_s {r.f_s_required/1e6:5.2f} MHz")

## Detector pulse shape per milk particle type

Top-hat particle convolved with the Gaussian beam (20 µm along the flow).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, spec in MILK_PARTICLES.items():
    rr = simulate(Params(particle_diameter=spec["d"]))
    ax.plot(rr.t * 1e6, rr.pulse, label=f"{name} ({units.to_um(spec['d']):.1f} µm), FWHM = {rr.fwhm*1e9:.0f} ns")
rr = simulate(Params(particle_diameter=um(20)))
ax.plot(rr.t * 1e6, rr.pulse, "--", label=f"large fat globule (20 µm), FWHM = {rr.fwhm*1e9:.0f} ns")
ax.set_xlabel("time [µs]")
ax.set_ylabel("normalised signal")
ax.legend()
ax.grid(alpha=0.3)

## Sample rate vs sheath flow

In [ ]:
q_sheath = np.linspace(2, 40, 40)  # mL/min
res = [simulate(Params(q_sheath=ml_per_min(q))) for q in q_sheath]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(q_sheath, [r.f_s_required / 1e6 for r in res])
ax1.axvline(20, color="k", ls=":", label="instrument")
ax1.set_xlabel("sheath flow [mL/min]"); ax1.set_ylabel("required sample rate [MHz]"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(q_sheath, [units.to_um(r.core_diameter) for r in res])
ax2.axhspan(*[units.to_um(x) for x in MEASURED_CORE_WIDTH_RANGE], alpha=0.15, label="measured 10-30 µm")
ax2.axvline(20, color="k", ls=":")
ax2.set_xlabel("sheath flow [mL/min]"); ax2.set_ylabel("core diameter [µm]"); ax2.legend(); ax2.grid(alpha=0.3)

## Scratch

Try new physics below; promote to the package when it settles.